In [1]:
import os
import mlflow

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:3900"
os.environ["AWS_ACCESS_KEY_ID"]      = "mlflow-access-key"
os.environ["AWS_SECRET_ACCESS_KEY"]  = "mlflow-secret-key"
os.environ["AWS_DEFAULT_REGION"]     = "garage"

mlflow.set_tracking_uri("http://localhost:5000/")

with mlflow.start_run():
    mlflow.log_param("param1", 15)
    mlflow.log_metric("metric1", 0.89)

/home/shaharyar/01__git-repos/devops-practice/28__mlops/00__Projects/youtube-sentiment-analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🏃 View run wise-ram-104 at: http://localhost:5000/#/experiments/0/runs/fd938421c1b549f28cdd7ae7a2eaa234
🧪 View experiment at: http://localhost:5000/#/experiments/0


In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [4]:
df.dropna(inplace=True)

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [7]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/shaharyar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/shaharyar/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [8]:
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

In [9]:
df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)

In [10]:
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [11]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
# Step 1: Vectorize the comments using Bag of Words (CountVectorizer)
vectorizer = CountVectorizer(max_features=10000)  # Bag of Words model with a limit of 1000 features

In [13]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']  # Assuming 'sentiment' is the target variable (0 or 1 for binary classification)

In [14]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(36799, 10000))

In [15]:
X.shape

(36799, 10000)

In [16]:
y

0        1
1        1
2       -1
3        0
4        1
        ..
37244    0
37245    1
37246    0
37247    1
37248    0
Name: category, Length: 36799, dtype: int64

In [17]:
y.shape

(36799,)

In [18]:
mlflow.set_tracking_uri("http://localhost:5000/")

In [19]:
mlflow.set_experiment("RF Baseline")

<Experiment: artifact_location='s3://mlflow-artifacts/1', creation_time=1777117015478, experiment_id='1', last_update_time=1777117015478, lifecycle_stage='active', name='RF Baseline', tags={}, trace_location=None, workspace='default'>

In [20]:
# Step 1: Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

with mlflow.start_run() as run:

    # ---------------------------
    # Tags / metadata
    # ---------------------------
    mlflow.set_tag("mlflow.runName", "RandomForest_Baseline_TrainTestSplit")
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag(
        "description",
        "Baseline RandomForest model for sentiment analysis using BoW (CountVectorizer) + train-test split"
    )

    # ---------------------------
    # Vectorizer params
    # ---------------------------
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

    # ---------------------------
    # Model params
    # ---------------------------
    n_estimators = 200
    max_depth = 15

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # ---------------------------
    # Train model
    # ---------------------------
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ---------------------------
    # Predictions
    # ---------------------------
    y_pred = model.predict(X_test)

    # ---------------------------
    # Metrics
    # ---------------------------
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    classification_rep = classification_report(y_test, y_pred, output_dict=True)

    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(f"{label}_{metric}", value)

    # ---------------------------
    # Confusion matrix plot
    # ---------------------------
    cm_path = "confusion_matrix.png"

    conf_matrix = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.tight_layout()

    plt.savefig(cm_path)
    plt.close()

    mlflow.log_artifact(cm_path)

    # ---------------------------
    # Log model
    # ---------------------------
    mlflow.sklearn.log_model(model, "random_forest_model")

    # ---------------------------
    # Save dataset artifact
    # ---------------------------
    data_path = "dataset.csv"
    df.to_csv(data_path, index=False)
    mlflow.log_artifact(data_path)

# Final output
print(f"Accuracy: {accuracy}")

2026/04/25 17:19:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:19:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_Baseline_TrainTestSplit at: http://localhost:5000/#/experiments/1/runs/6731c1b42924433f8b518e8b1a2e1d0e
🧪 View experiment at: http://localhost:5000/#/experiments/1
Accuracy: 0.6452445652173913


In [21]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          -1       1.00      0.01      0.02      1650
           0       0.67      0.81      0.74      2556
           1       0.62      0.84      0.72      3154

    accuracy                           0.65      7360
   macro avg       0.77      0.55      0.49      7360
weighted avg       0.73      0.65      0.57      7360



In [22]:
df.to_csv('reddit_preprocessing.csv', index=False)

In [23]:
pd.read_csv('reddit_preprocessing.csv').head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1
